eventuele, verbeterpunten:
- kijken welke batch size het beste is
- Activatiefunctie is nu globaal → je kan ook per laag variëren:
- Dropout (tegen overfitting)
- BatchNormalization

lijst van parameters die aangepast worden met optimization:
- aantal dense lagen: `'num_layers', 3, 10`
- aantal units: `'units_{i}', min_value=32, max_value=512, step=32`
- activation function: `'activation', ['relu', 'tanh']`
- learning rate: `'learning_rate', [1e-5, 1e-4, 1e-3, 1e-2]`

lijst van gekozen parameters:
- a
- a
- a
- a
- a
- a
- a

# GOED

In [8]:
import tensorflow as tf
from tensorflow import keras
from kerastuner.tuners import BayesianOptimization
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

# GOED

In [9]:
def make_sequences(data, window_size=7, output_steps=30): # window_size = hoe ver in de toekomst wil je voorspellen
    X = []
    for i in range(len(data) - window_size - output_steps + 1):
        X.append(data[i:i+window_size])
    return np.array(X)

In [10]:
def make_targets(data, window_size=7, output_steps=30): # window_size = hoe ver in de toekomst wil je voorspellen
    y = []
    for i in range(len(data) - window_size - output_steps + 1):
        y.append(data[i+window_size:i+window_size+output_steps])
    return np.array(y)

In [12]:
def make_dataset(train_dfs):
    X_datasets = []
    y_datasets = []
    for train_df in train_dfs:
        dfy = train_df[["matches"]].values
        dfx = train_df.drop(columns=["poi_id", "matches"]).values
        X_train = make_sequences(dfx)
        y_train = make_targets(dfy)

        X_datasets.append(X_train)
        y_datasets.append(y_train)

    X_dataset = X_datasets[0]
    for ds in X_datasets[1:]:
        X_dataset = np.concatenate([X_dataset, ds])

    y_dataset = y_datasets[0]
    for ds in y_datasets[1:]:
        y_dataset = np.concatenate([y_dataset, ds])
    
    return X_dataset, y_dataset

In [13]:
# 1 laad de dataset in
df = pd.read_csv("../data_cleaned/RNN_dataset.csv")

# poi_id string maken -> hotonencoding
df["poi_id"] = df["poi_id"].astype(str)

df["date"] = pd.to_datetime(df["date"])

# 5 maak een kopie van de target kolom
target_kolom = df[["poi_id", "matches", "date"]].copy()

# 3 verdeel je dataset in train_df, val_df en test_df
temp = df[df["date"] <= '2025-11-15']
train_df = temp[temp["date"] <= '2025-09-15']
val_df = temp[temp["date"] >= '2025-09-16']
test_df = df[df["date"] >= '2025-11-16']

train_df = train_df.drop(columns=["date"])
val_df = val_df.drop(columns=["date"])
test_df = test_df.drop(columns=["date"])

# 2 maak de standardscaler en hotencoding pipeline
categorical_cols = ["poi_id", "month", "day_name", "Seizoen"]
numerical_cols = [col for col in train_df.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# 4 hotencode en standardscale de dfs
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

# 6 verdeel je gekopieerde kolom in y_train, y_val en y_test
# 3 verdeel je dataset in train_df, val_df en test_df
y_temp_df = target_kolom[target_kolom["date"] <= '2025-11-15']
y_train = y_temp_df[y_temp_df["date"] <= '2025-09-15']
y_val = y_temp_df[y_temp_df["date"] >= '2025-09-16']
y_test = target_kolom[target_kolom["date"] >= '2025-11-16']

y_train = y_train.drop(columns=["date"])
y_val = y_val.drop(columns=["date"])
y_test = y_test.drop(columns=["date"])

# maak X_train een dataset en voeg X_train en y_train samen -> dit ook van val en test
all_cols = preprocessor.get_feature_names_out()
df_X_train_transformed = pd.DataFrame(X_train, columns=all_cols)
y_train = y_train.reset_index(drop=True)
df_X_train_transformed = df_X_train_transformed.reset_index(drop=True)
concat_train = pd.concat([y_train, df_X_train_transformed], axis=1)

df_X_val_transformed = pd.DataFrame(X_val, columns=all_cols)
y_val = y_val.reset_index(drop=True)
df_X_val_transformed = df_X_val_transformed.reset_index(drop=True)
concat_val = pd.concat([y_val, df_X_val_transformed], axis=1)

df_X_test_transformed = pd.DataFrame(X_test, columns=all_cols)
y_test = y_test.reset_index(drop=True)
df_X_test_transformed = df_X_test_transformed.reset_index(drop=True)
concat_test = pd.concat([y_test, df_X_test_transformed], axis=1)

# splits de dataset per poi_id
train_dfs = [group for _, group in concat_train.groupby("poi_id")]
val_dfs = [group for _, group in concat_val.groupby("poi_id")]
test_dfs = [group for _, group in concat_test.groupby("poi_id")]

X_train, y_train = make_dataset(train_dfs)
X_val, y_val = make_dataset(val_dfs)
X_test, y_test = make_dataset(test_dfs)

In [15]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(27084, 7, 184)
(27084, 30, 1)
(11956, 7, 184)
(11956, 30, 1)
(3050, 7, 184)
(3050, 30, 1)


In [35]:
X_train.shape

(27084, 7, 184)

In [36]:
X_train.shape[1]

7

# eventueel parameters aanpassen
model.add(keras.Input(shape=(X_train.shape[1], 1)))  aanpassen

In [3]:
# --- 2. Modelbuilder functie ---
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(7, 184)))  # shape: (timesteps, features) # AANPASSEN
    # model.add(norm_layer)

    # zodat laatste laag zelfde is als vorige lagen
    activation_function = hp.Choice('activation', ['relu', 'tanh'])
    for i in range(hp.Int('num_layers', 2, 10)):
        model.add(
            keras.layers.SimpleRNN(
                units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
                activation=activation_function,
                return_sequences=True
            )
        )
    
    # laatste laag is return_sequence false
    model.add(
            keras.layers.SimpleRNN(
                units=hp.Int(f'laatste_unit', min_value=32, max_value=512, step=32),
                activation=activation_function,
                return_sequences=False
            )
        )
    
    model.add(keras.layers.Dense(30))  # outputlaag voor regressie

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Choice('learning_rate', [1e-5, 1e-4, 1e-3, 1e-2])
        ),
        loss='mse', # wat hij gaat proberen verminderen
        metrics=['mse', 'RootMeanSquaredError'] # dit is puur voor visueel zodat je kan interpreteren hoe het model presteert
    )
    
    return model

# max trials? objective? num_initial_points?

In [16]:
# --- 3. Bayesian Optimization Tuner ---
tuner = BayesianOptimization(
    build_model,
    objective='val_mse', # wat hij naar beneden wilt krijgen
    max_trials=60,  # max aantal hyperparametercombinaties om te proberen
    num_initial_points=20, # hoeveel random hyperparametercombinaties uitproberen voordat hij bayesion optimization toepast?
    directory='../Bachelorproef_modellen/RNN_bay',
    project_name='RNN_bayian_opt'
)

Reloading Tuner from ../Bachelorproef_modellen/RNN_bay\RNN_bayian_opt\tuner0.json


# eventueel patience, batch_size

In [39]:
# --- 4. Start hyperparameter tuning ---
early_stopping_cb = keras.callbacks.EarlyStopping(
    monitor='val_loss',       # validatie MSE
    patience=10,
    min_delta=0.001,         # kleine verbetering in MAE is al goed
    restore_best_weights=True,
    mode='min'               # MAE moet omlaag
)

tuner.search(
    X_train, y_train,
    epochs=200,
    validation_data=(X_val, y_val),
    batch_size=32, # batch_size=hp.Choice('batch_size', [16, 32, 64]) ?
    callbacks=[early_stopping_cb],
    verbose=1
)

Trial 60 Complete [00h 02m 46s]
val_mse: 8.11136245727539

Best val_mse So Far: 7.929819583892822
Total elapsed time: 05h 18m 47s


# GOED

In [17]:
# --- 5. Beste hyperparameters en model ---
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Beste hyperparameters:")
print(best_hp.values)

best_model = tuner.get_best_models(1)[0]

Beste hyperparameters:
{'activation': 'relu', 'num_layers': 2, 'units_0': 32, 'units_1': 256, 'laatste_unit': 512, 'learning_rate': 0.01, 'units_2': 32, 'units_3': 32, 'units_4': 256, 'units_5': 64, 'units_6': 32, 'units_7': 96, 'units_8': 128, 'units_9': 192}


In [18]:
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn (SimpleRNN)      (None, 7, 32)             6944      
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 7, 256)            73984     
                                                                 
 simple_rnn_2 (SimpleRNN)    (None, 512)               393728    
                                                                 
 dense (Dense)               (None, 30)                15390     
                                                                 
Total params: 490046 (1.87 MB)
Trainable params: 490046 (1.87 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
results = best_model.evaluate(X_test, y_test)
print(f"general loss: {results[0]}")
print(f"general mse: {results[1]}")
print(f"general rmse: {results[2]}")

374/374 [==============================] - 3s 6ms/step - loss: 8.7821 - mse: 8.7821 - root_mean_squared_error: 2.9635
general loss: 8.782120704650879
general mse: 8.782120704650879
general rmse: 2.9634640216827393


In [20]:
best_model = tuner.get_best_models(1)[0]
# pipeline = onehote encoding, standardizing, decisiontree training ---> trainingsdata
predictions = best_model.predict(X_test)
# pipeline = onehote encoding, standardizing, decisiontree voorspel ---> input

# testen op under en overfitting
train_predictions = best_model.predict(X_train)
val_predictions = best_model.predict(X_val)
print(f"Training:  {mean_absolute_percentage_error(y_train.flatten(), train_predictions.flatten()):2%}")
print(f"Validatie:  {mean_absolute_percentage_error(y_val.flatten(), val_predictions.flatten()):.2%}")


# Evaluatie
mae = mean_absolute_error(y_test.flatten(), predictions.flatten())
rmse = root_mean_squared_error(y_test.flatten(), predictions.flatten())
mape = mean_absolute_percentage_error(y_test.flatten(), predictions.flatten())
r2 = r2_score(y_test.flatten(), predictions.flatten())

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2%}")
print(f"R²: {r2:.2f}")

# # Visualiseer en interpreteer je resultaten.
# plt.scatter(y_test, predictions, alpha=0.7)
# plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
# plt.xlabel('Werkelijke Kijkcijfers')
# plt.ylabel('Voorspelde Kijkcijfers')
# plt.title('Werkelijke vs Voorspelde Kijkcijfers')
# plt.show()

96/96 [==============================] - 1s 6ms/step
Training:  50.893972%
Validatie:  50.83%
MAE: 2.17
RMSE: 2.96
MAPE: 53.58%
R²: 0.45


# EXPERIMENTEN

## termijnen

In [26]:
# Evaluatie
kt_mae = mean_absolute_error(y_test[:, 0:7].flatten(), predictions[:, 0:7].flatten())
kt_rmse = root_mean_squared_error(y_test[:, 0:7].flatten(), predictions[:, 0:7].flatten())
kt_r2 = r2_score(y_test[:, 0:7].flatten(), predictions[:, 0:7].flatten())

mt_mae = mean_absolute_error(y_test[:, 7:21].flatten(), predictions[:, 7:21].flatten())
mt_rmse = root_mean_squared_error(y_test[:, 7:21].flatten(), predictions[:, 7:21].flatten())
mt_r2 = r2_score(y_test[:, 7:21].flatten(), predictions[:, 7:21].flatten())

lt_mae = mean_absolute_error(y_test[:, 21:30].flatten(), predictions[:, 21:30].flatten())
lt_rmse = root_mean_squared_error(y_test[:, 21:30].flatten(), predictions[:, 21:30].flatten())
lt_r2 = r2_score(y_test[:, 21:30].flatten(), predictions[:, 21:30].flatten())

print(f"Korte termijn MAE: {kt_mae:.2f}")
print(f"Korte termijn RMSE: {kt_rmse:.2f}")
print(f"Korte termijn R²: {kt_r2:.2f}")
print()

print(f"Midden termijn MAE: {mt_mae:.2f}")
print(f"Midden termijn RMSE: {mt_rmse:.2f}")
print(f"Midden termijn R²: {mt_r2:.2f}")
print()

print(f"Lange termijn MAE: {lt_mae:.2f}")
print(f"Lange termijn RMSE: {lt_rmse:.2f}")
print(f"Lange termijn R²: {lt_r2:.2f}")

Korte termijn MAE: 2.13
Korte termijn RMSE: 2.89
Korte termijn R²: 0.47

Midden termijn MAE: 2.17
Midden termijn RMSE: 2.97
Midden termijn R²: 0.45

Lange termijn MAE: 2.20
Lange termijn RMSE: 3.01
Lange termijn R²: 0.45
